# OpenMontage Stage 2: Colab GPU lightweight adapter checks
This notebook performs safe, one-shot diagnostics for optional Stage 2 adapters (music, tts, transcriber, enhancement) on a fresh Colab GPU runtime.
DO NOT run the full pipeline. The notebook: clones the repo, sets OM_LOAD_ADAPTERS=1, checks CUDA, attempts lightweight imports/instantiation, runs one small transcriber test, runs safe enhancement passthrough, and runs conservative TTS/music checks.
All diagnostics are written to projects/smoke-report/artifacts/real_init.log.


In [ ]:
# Cell 1: Setup repository and environment (run once)
import os, sys, subprocess
REPO_DIR = '/content/openmontage-colab'
if os.path.exists(REPO_DIR):
    print('Repo exists, pulling latest on refactor/colab-ready')
    subprocess.run(['bash','-lc', f'cd {REPO_DIR} && git fetch --all && git checkout refactor/colab-ready && git pull origin refactor/colab-ready'], check=False)
else:
    print('Cloning repository and checking out refactor/colab-ready')
    subprocess.run(['bash','-lc', f'git clone https://github.com/jvvghj123-sudo/openmontage-colab.git {REPO_DIR}'], check=True)
    subprocess.run(['bash','-lc', f'cd {REPO_DIR} && git checkout refactor/colab-ready'], check=False)
# Add repo to PYTHONPATH and set runtime env vars
sys.path.insert(0, REPO_DIR)
os.environ['OM_LOAD_ADAPTERS'] = '1'
# For Stage 2 optional checks, REAL_STRICT not required; keep clear: we set LOAD_ADAPTERS only
print('OM_LOAD_ADAPTERS=1 set; repo path added to PYTHONPATH:', REPO_DIR)


In [ ]:
# Cell 2: CUDA / GPU check
import torch
print('torch version:', getattr(torch, '__version__', 'n/a'))
cuda_ok = torch.cuda.is_available()
print('CUDA available:', cuda_ok)
if cuda_ok:
    try:
        print('Device:', torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print('VRAM GB:', round(props.total_memory/(1024**3),2))
    except Exception as e:
        print('Device query failed:', e)
else:
    print('No CUDA device detected in this runtime — real GPU inference cannot run here.')


In [ ]:
# Cell 3: Adapter availability and lightweight instantiation
import json, traceback, os
from src.adapters.adapter_loader import MODEL_TO_ADAPTER, get_component
from src.registry import load_registry
reg = load_registry().data
roles = ['music','tts','transcriber','enhancement']
summary = {}
art_dir = os.path.join('projects','smoke-report','artifacts')
os.makedirs(art_dir, exist_ok=True)
for role in roles:
    entry = {'role': role}
    try:
        choice = reg.get(role, {}).get('default_choice')
        entry['choice'] = choice
        mapping = MODEL_TO_ADAPTER.get(choice)
        entry['mapping'] = mapping
        # Use get_component to respect loader logic (it imports only if OM_LOAD_ADAPTERS=1)
        comp = None
        try:
            comp = get_component(role, profile='balanced')
            entry['instantiated'] = comp is not None
            entry['class'] = comp.__class__.__name__ if comp else None
            entry['available_flag'] = getattr(comp, 'available', None) if comp else None
        except Exception as e:
            entry['instantiated'] = False
            entry['inst_error'] = str(e)
            entry['inst_traceback'] = traceback.format_exc()
    except Exception as e:
        entry['error'] = str(e)
        entry['traceback'] = traceback.format_exc()
    summary[role] = entry
# Write summary to artifacts and print
with open(os.path.join(art_dir,'stage2_adapter_summary.json'),'w',encoding='utf-8') as f:
    f.write(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print('
Diagnostics appended to', os.path.join(art_dir,'stage2_adapter_summary.json'))


In [ ]:
# Cell 4: Transcriber one-shot test (small model)
# Creates a 1s silent WAV and runs the transcriber using a small model.
import os, traceback
from pathlib import Path
p = Path('projects/colab-stage2-test')
p.mkdir(parents=True, exist_ok=True)
wav_path = p/'silence_short.wav'
try:
    import numpy as np
    from scipy.io.wavfile import write as wavwrite
    sr = 16000
    silence = np.zeros(sr, dtype=np.int16)
    wavwrite(str(wav_path), sr, silence)
    print('Created:', wav_path)
except Exception as e:
    print('Failed to create WAV:', e)
# Instantiate transcriber adapter class directly with small model to avoid huge downloads where possible
try:
    from src.adapters import adapter_loader
    mapping = adapter_loader.MODEL_TO_ADAPTER.get('faster-whisper')
    if mapping:
        mod_name, cls_name = mapping
        mod = __import__(f'src.adapters.{mod_name}', fromlist=['*'])
        cls = getattr(mod, cls_name)
        # pass config to use small model spec
        inst = cls('faster-whisper', config={'model': 'small'})
        print('Transcriber instantiated:', inst.__class__.__name__, 'available=', getattr(inst,'available',None))
        if not getattr(inst,'available',False):
            print('Transcriber backend not available locally — TEST SKIPPED')
        else:
            try:
                res = inst.run(str(wav_path))
                print('Transcription (first 200 chars):', res.get('transcript')[:200])
                print('PASS: Real transcriber produced output')
            except Exception as e:
                print('Transcriber run failed (real model test failed):', e)
    else:
        print('No mapping for faster-whisper; skipping')
except Exception as e:
    print('Transcriber instantiation error:', traceback.format_exc())


In [ ]:
# Cell 5: Enhancement test (safe)
# Instantiate enhancement adapter and run on a tiny image. Do NOT install heavy packages here.
from pathlib import Path
from PIL import Image
from src.adapters.adapter_loader import get_component
p = Path('projects/colab-stage2-test')
p.mkdir(parents=True, exist_ok=True)
img_p = p/'small.png'
Image.new('RGB', (64,64), (128,128,128)).save(img_p)
inst = get_component('enhancement', profile='balanced')
if not inst:
    print('Enhancement adapter not available (skipping)')
else:
    print('Enhancement adapter class:', inst.__class__.__name__, 'available=', getattr(inst,'available',None))
    try:
        out = inst.run(str(img_p))
        print('Enhancement run result:', out)
        if out.get('status') == 'success':
            print('PASS: enhancement executed with real backend')
        else:
            print('INFO: enhancement returned passthrough or mock result — not a real model run')
    except Exception as e:
        print('Enhancement execution failed:', e)


In [ ]:
# Cell 6: Music and TTS lightweight checks
from src.adapters.adapter_loader import get_component
from pathlib import Path
p = Path('projects/colab-stage2-test')
p.mkdir(parents=True, exist_ok=True)
# Music: instantiate but do NOT generate audio (can be heavy).
m = get_component('music', profile='balanced')
print('Music adapter:', m.__class__.__name__ if m else None, 'available=', getattr(m,'available',None) if m else None)
if m and getattr(m,'available',False):
    print('Music adapter available — skipping full generation to avoid heavy downloads. To validate, run one-shot on Colab GPU after confirming vram.')
# TTS: try to synthesize a short fallback audio (may use local pyttsx3 or scipy fallback)
tts = get_component('tts', profile='balanced')
print('TTS adapter:', tts.__class__.__name__ if tts else None)
if tts:
    try:
        out = tts.run('Stage 2 test TTS', output=str(p/'tts.wav'))
        print('TTS run result:', out)
        if out.get('status') == 'success' and out.get('output_path'):
            print('PASS: TTS produced a real audio file at', out.get('output_path'))
        else:
            print('INFO: TTS returned fallback or mock output — considered NOT a real model test')
    except Exception as e:
        print('TTS run failed or used fallback:', e)
else:
    print('No TTS adapter instantiated')


In [ ]:
# Cell 7: Final summary and where logs are saved
print('Stage 2 checks complete.')
print('Review artifacts: projects/smoke-report/artifacts/real_init.log and projects/smoke-report/artifacts/stage2_adapter_summary.json')
import os
print('Artifact files exist:')
for f in ['projects/smoke-report/artifacts/real_init.log','projects/smoke-report/artifacts/stage2_adapter_summary.json']:
    print(f, ':', os.path.exists(f))
